# 03 - Auditoría XAI: SHAP, modelo subrogado y contrafactuales

Este notebook explica modelos que, por construcción, no son transparentes:

- Red neuronal MLP.
- Política neuronal de contextual bandit.

La auditoría se organiza en tres niveles:

1. **Explicación global**: qué variables son importantes en promedio.
2. **Explicación local**: por qué un cliente concreto recibe una predicción.
3. **Contrafactuales**: qué tendría que cambiar para modificar la decisión.

También se entrena un **modelo subrogado** interpretable: un árbol de decisión pequeño que intenta imitar las decisiones del modelo caja negra. Esto permite extraer reglas aproximadas.

In [ ]:
# ==============================
# Imports
# ==============================

from __future__ import annotations

import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import joblib

# Necesario para poder reconstruir ScaledKNNImputer al cargar
# preprocessing_objects.joblib (ver preprocessing_utils.py).
from preprocessing_utils import ScaledKNNImputer  # noqa: F401
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

try:
    import shap
    SHAP_AVAILABLE = True
except Exception as exc:
    SHAP_AVAILABLE = False
    print("SHAP no está disponible. Se usará importancia por permutación como fallback.")
    print(exc)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

OUTPUT_DIR = Path("outputs")
OBJECTS_DIR = OUTPUT_DIR / "objects"
MODELS_DIR = OUTPUT_DIR / "models"
XAI_DIR = OUTPUT_DIR / "xai"
XAI_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Carga de datos, modelos y metadatos

Este notebook requiere haber ejecutado:

1. `01_EDA_Preprocesado.ipynb`
2. `02_Modelado_RL_NN.ipynb`

Se cargan los modelos finales y se reconstruyen las clases PyTorch con la misma arquitectura usada durante entrenamiento.

In [ ]:
required_files = [
    OUTPUT_DIR / "preprocessed_train.csv",
    OBJECTS_DIR / "preprocessing_objects.joblib",
    OBJECTS_DIR / "final_model_scaler.joblib",
    MODELS_DIR / "model_metadata.joblib",
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(f"Falta {path}. Ejecuta primero los notebooks 01 y 02.")

train_df = pd.read_csv(OUTPUT_DIR / "preprocessed_train.csv")
preprocessing_objects = joblib.load(OBJECTS_DIR / "preprocessing_objects.joblib")
scaler = joblib.load(OBJECTS_DIR / "final_model_scaler.joblib")
model_metadata = joblib.load(MODELS_DIR / "model_metadata.joblib")

TARGET = preprocessing_objects["target"]
FEATURES = preprocessing_objects["final_features"]

X = train_df[FEATURES].astype(float)
y = train_df[TARGET].astype(int)
X_scaled = scaler.transform(X)

print("Datos:", X.shape)
print("Modelos seleccionados:")
print(json.dumps(model_metadata["selected_models"], indent=2))

In [ ]:
class CreditMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims=(128, 64, 32), dropout: float = 0.25):
        super().__init__()
        h1, h2, h3 = hidden_dims
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h2, h3),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(h3, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


class ContextualBanditPolicy(nn.Module):
    def __init__(self, input_dim: int, hidden_dims=(128, 64), dropout: float = 0.20):
        super().__init__()
        h1, h2 = hidden_dims
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.BatchNorm1d(h1),
            nn.Dropout(dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.BatchNorm1d(h2),
            nn.Dropout(dropout),
            nn.Linear(h2, 2),
        )

    def forward(self, x):
        return self.net(x)


def load_torch_model(model_path: Path):
    """Carga un modelo PyTorch guardado en el notebook 02."""
    checkpoint = torch.load(model_path, map_location=DEVICE)
    if checkpoint["model_type"] == "MLP":
        model = CreditMLP(
            input_dim=checkpoint["input_dim"],
            hidden_dims=tuple(checkpoint["hidden_dims"]),
            dropout=checkpoint["dropout"],
        )
    elif checkpoint["model_type"] == "ContextualBanditPolicy":
        model = ContextualBanditPolicy(
            input_dim=checkpoint["input_dim"],
            hidden_dims=tuple(checkpoint["hidden_dims"]),
            dropout=checkpoint["dropout"],
        )
    else:
        raise ValueError(f"Tipo de modelo no reconocido: {checkpoint['model_type']}")

    model.load_state_dict(checkpoint["state_dict"])
    model.to(DEVICE)
    model.eval()
    return model, checkpoint


def predict_model_score(model, model_family: str, X_array: np.ndarray, batch_size: int = 4096) -> np.ndarray:
    """
    Devuelve score comparable a probabilidad de clase 1.

    - MLP: sigmoid(logit).
    - Bandit: probabilidad de acción 1 por softmax.
    """
    scores = []
    with torch.no_grad():
        for start in range(0, len(X_array), batch_size):
            xb = torch.tensor(X_array[start:start + batch_size], dtype=torch.float32, device=DEVICE)
            if model_family == "MLP":
                score = torch.sigmoid(model(xb)).cpu().numpy()
            elif model_family == "ContextualBanditPolicy":
                score = F.softmax(model(xb), dim=1)[:, 1].cpu().numpy()
            else:
                raise ValueError(model_family)
            scores.append(score)
    return np.concatenate(scores)


def predict_selected_scenario(scenario: str, X_raw: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, Dict]:
    """Predice score y clase para el modelo seleccionado de un escenario."""
    selected = model_metadata["selected_models"][scenario]
    model_path = MODELS_DIR / selected["model_file"]
    model, checkpoint = load_torch_model(model_path)
    X_arr = scaler.transform(X_raw[FEATURES].astype(float))
    score = predict_model_score(model, selected["model_family"], X_arr)
    pred = (score >= selected["threshold"]).astype(int)
    return score, pred, selected

In [ ]:
# Calculamos predicciones para ambos escenarios.
scenario_predictions = {}
for scenario in model_metadata["scenarios"].keys():
    score, pred, selected = predict_selected_scenario(scenario, X)
    scenario_predictions[scenario] = {"score": score, "pred": pred, "selected": selected}
    print("\nEscenario:", scenario)
    print("Modelo:", selected)
    print("AUC aproximado:", roc_auc_score(y, score))
    display(pd.DataFrame(confusion_matrix(y, pred, labels=[0, 1]), index=["real_0", "real_1"], columns=["pred_0", "pred_1"]))

## 2. Selección del modelo a auditar

Auditaremos principalmente el escenario `cost_1_10`, porque es el más sensible desde el punto de vista de riesgo: equivocarse concediendo crédito a un cliente que acaba en mora grave cuesta diez veces más que rechazar a un cliente bueno.

Aun así, el código permite cambiar `AUDIT_SCENARIO` a `cost_1_1`.

In [ ]:
AUDIT_SCENARIO = "cost_1_10"

audit_score = scenario_predictions[AUDIT_SCENARIO]["score"]
audit_pred = scenario_predictions[AUDIT_SCENARIO]["pred"]
audit_selected = scenario_predictions[AUDIT_SCENARIO]["selected"]

audit_model, audit_checkpoint = load_torch_model(MODELS_DIR / audit_selected["model_file"])
audit_family = audit_selected["model_family"]
audit_threshold = audit_selected["threshold"]

print("Escenario auditado:", AUDIT_SCENARIO)
print("Modelo auditado:", audit_selected)

## 3. Modelo subrogado interpretable

Un modelo subrogado intenta imitar el comportamiento del modelo caja negra, no el target real directamente.

Aquí entrenamos un árbol de decisión pequeño con `max_depth=3` para aproximar las decisiones del modelo auditado. El objetivo es extraer reglas sencillas del tipo:

> Si el número de retrasos de 90 días es alto y la utilización de crédito es alta, entonces el modelo tiende a clasificar como alto riesgo.

La métrica importante del subrogado es la **fidelidad**, es decir, cuánto coincide con la caja negra.

In [ ]:
# Dividimos para medir fidelidad fuera de la muestra usada por el subrogado.
X_sur_train, X_sur_test, y_sur_train, y_sur_test = train_test_split(
    X,
    audit_pred,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=audit_pred,
)

surrogate_tree = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=300,
    random_state=RANDOM_STATE,
)
surrogate_tree.fit(X_sur_train, y_sur_train)

sur_pred = surrogate_tree.predict(X_sur_test)
fidelity = accuracy_score(y_sur_test, sur_pred)

print("Fidelidad del árbol subrogado respecto al modelo caja negra:", round(fidelity, 4))
print(classification_report(y_sur_test, sur_pred, digits=4, zero_division=0))

In [ ]:
plt.figure(figsize=(24, 10))
plot_tree(
    surrogate_tree,
    feature_names=FEATURES,
    class_names=["pred_0", "pred_1"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=9,
)
plt.title(f"Árbol subrogado del modelo auditado - {AUDIT_SCENARIO}")
plt.show()

In [ ]:
rules_text = export_text(surrogate_tree, feature_names=FEATURES, decimals=3)
print(rules_text)

with open(XAI_DIR / f"surrogate_rules_{AUDIT_SCENARIO}.txt", "w", encoding="utf-8") as f:
    f.write(rules_text)

print("Reglas guardadas en:", XAI_DIR / f"surrogate_rules_{AUDIT_SCENARIO}.txt")

## 4. Importancia global por permutación

La importancia por permutación mide cuánto empeora el modelo si rompemos aleatoriamente una variable. Si al permutar una variable el score del modelo empeora mucho, significa que el modelo dependía bastante de esa variable.

Para evitar tiempos excesivos, usamos una muestra.

In [ ]:
class BlackBoxWrapper:
    """
    Wrapper estilo sklearn para poder usar permutation_importance.
    """
    def __init__(self, model, model_family: str, threshold: float):
        self.model = model
        self.model_family = model_family
        self.threshold = threshold

    def predict_proba(self, X_input):
        if isinstance(X_input, pd.DataFrame):
            X_arr = scaler.transform(X_input[FEATURES].astype(float))
        else:
            X_arr = scaler.transform(pd.DataFrame(X_input, columns=FEATURES).astype(float))
        score = predict_model_score(self.model, self.model_family, X_arr)
        return np.vstack([1 - score, score]).T

    def predict(self, X_input):
        return (self.predict_proba(X_input)[:, 1] >= self.threshold).astype(int)

    def score(self, X_input, y_input):
        return roc_auc_score(y_input, self.predict_proba(X_input)[:, 1])


blackbox = BlackBoxWrapper(audit_model, audit_family, audit_threshold)

perm_sample_idx = X.sample(n=min(8000, len(X)), random_state=RANDOM_STATE).index
X_perm = X.loc[perm_sample_idx]
y_perm = y.loc[perm_sample_idx]

perm = permutation_importance(
    blackbox,
    X_perm,
    y_perm,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="roc_auc",
)

perm_importance = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

perm_importance.head(20)

In [ ]:
plt.figure(figsize=(10, 7))
top_perm = perm_importance.head(20).iloc[::-1]
plt.barh(top_perm["feature"], top_perm["importance_mean"])
plt.title(f"Importancia por permutación - {AUDIT_SCENARIO}")
plt.xlabel("Disminución media en ROC AUC al permutar")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

perm_importance.to_csv(XAI_DIR / f"permutation_importance_{AUDIT_SCENARIO}.csv", index=False)

## 5. SHAP global y local

SHAP descompone la predicción en aportaciones por variable. Para modelos neuronales tabulares, `KernelExplainer` es más general pero más lento. Por eso usamos muestras pequeñas:

- `background`: muestra usada como referencia.
- `explain_sample`: clientes concretos que explicamos.

Si el equipo tiene poco tiempo de cómputo, se puede reducir el tamaño de las muestras.

In [ ]:
# Función que SHAP puede llamar: recibe datos en escala original y devuelve probabilidad de clase 1.
def shap_predict_proba(x_numpy: np.ndarray) -> np.ndarray:
    x_df = pd.DataFrame(x_numpy, columns=FEATURES)
    x_scaled = scaler.transform(x_df.astype(float))
    return predict_model_score(audit_model, audit_family, x_scaled)

# Muestras para SHAP.
background = X.sample(n=min(100, len(X)), random_state=RANDOM_STATE)
explain_sample = X.sample(n=min(300, len(X)), random_state=RANDOM_STATE + 1)

print("Background SHAP:", background.shape)
print("Muestra a explicar:", explain_sample.shape)

In [ ]:
if SHAP_AVAILABLE:
    # KernelExplainer es lento pero válido para cualquier caja negra.
    explainer = shap.KernelExplainer(shap_predict_proba, background)
    shap_values = explainer.shap_values(explain_sample, nsamples=100)

    # En algunas versiones shap_values puede venir como lista; normalizamos.
    if isinstance(shap_values, list):
        shap_values_array = shap_values[0]
    else:
        shap_values_array = shap_values

    print("SHAP values shape:", np.array(shap_values_array).shape)
else:
    shap_values_array = None

In [ ]:
if SHAP_AVAILABLE and shap_values_array is not None:
    shap.summary_plot(shap_values_array, explain_sample, feature_names=FEATURES, show=True)

In [ ]:
if SHAP_AVAILABLE and shap_values_array is not None:
    shap.summary_plot(shap_values_array, explain_sample, feature_names=FEATURES, plot_type="bar", show=True)

In [ ]:
if SHAP_AVAILABLE and shap_values_array is not None:
    shap_global = pd.DataFrame({
        "feature": FEATURES,
        "mean_abs_shap": np.abs(shap_values_array).mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False)
    display(shap_global.head(20))
    shap_global.to_csv(XAI_DIR / f"shap_global_{AUDIT_SCENARIO}.csv", index=False)
else:
    shap_global = perm_importance.rename(columns={"importance_mean": "mean_abs_shap"})[["feature", "mean_abs_shap"]]

### 5.1 Explicaciones locales

Ahora seleccionamos varios clientes:

- Algunos de clase real 0.
- Algunos de clase real 1.
- Dentro de cada grupo, priorizamos casos donde el modelo da una decisión clara.

Esto sirve para responder a preguntas del tipo:

> ¿Por qué se me ha denegado el crédito?

In [ ]:
# Elegimos ejemplos reales de cada clase.
examples_0 = train_df[y == 0].copy()
examples_1 = train_df[y == 1].copy()

examples_0["score"] = audit_score[y == 0]
examples_1["score"] = audit_score[y == 1]

# Casos reales 0 con score alto son interesantes porque podrían ser rechazados pese a no haber incumplido.
selected_0_idx = examples_0.sort_values("score", ascending=False).head(3).index.tolist()

# Casos reales 1 con score alto muestran explicaciones de riesgo claro.
selected_1_idx = examples_1.sort_values("score", ascending=False).head(3).index.tolist()

local_indices = selected_0_idx + selected_1_idx
local_cases = X.loc[local_indices].copy()
local_scores = audit_score[local_indices]
local_preds = audit_pred[local_indices]
local_true = y.loc[local_indices].values

local_summary = pd.DataFrame({
    "index": local_indices,
    "true_class": local_true,
    "model_score_class_1": local_scores,
    "model_prediction": local_preds,
})
local_summary

In [ ]:
if SHAP_AVAILABLE:
    local_shap_values = explainer.shap_values(local_cases, nsamples=200)
    if isinstance(local_shap_values, list):
        local_shap_values = local_shap_values[0]

    for i, idx in enumerate(local_indices):
        print("\nCaso index:", idx, "| real:", local_true[i], "| pred:", local_preds[i], "| score:", round(local_scores[i], 4))
        contrib = pd.DataFrame({
            "feature": FEATURES,
            "value": local_cases.iloc[i].values,
            "shap_value": local_shap_values[i],
            "abs_shap": np.abs(local_shap_values[i]),
        }).sort_values("abs_shap", ascending=False)
        display(contrib.head(10))
else:
    print("SHAP no disponible; usa la importancia por permutación y el subrogado para explicación global/local.")

## 6. Contrafactuales

Un contrafactual responde:

> ¿Qué tendría que cambiar para que el modelo cambiara su decisión?

Ejemplo: si el modelo deniega crédito, buscamos cambios razonables que hagan que pase a aprobarlo.

Implementamos una búsqueda sencilla y transparente:

1. Tomamos un cliente concreto.
2. Modificamos una variable cada vez hacia valores percentiles observados en construcción.
3. Probamos combinaciones greedy de cambios.
4. Devolvemos el primer caso que cruza el threshold.

No todas las variables son igualmente modificables. Por ejemplo, la edad no es una recomendación accionable. Por eso definimos una lista de variables candidatas más razonables.

In [ ]:
# Variables originales que permitimos modificar en los contrafactuales.
# Evitamos cambiar directamente variables derivadas o logarítmicas, porque eso generaría perfiles incoherentes.
MUTABLE_FEATURES = [
    "RevolvingUtilizationOfUnsecuredLines",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberRealEstateLoansOrLines",
    "NumberOfDependents",
]

MUTABLE_FEATURES = [c for c in MUTABLE_FEATURES if c in FEATURES]

# Para proponer cambios, usamos percentiles reales observados en construcción.
percentile_grid = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
feature_quantiles = {
    col: X[col].quantile(percentile_grid).values
    for col in MUTABLE_FEATURES
}


def recompute_engineered_features_for_row(row: pd.Series) -> pd.Series:
    """
    Recalcula variables derivadas para mantener coherencia tras modificar una variable original.
    """
    out = row.copy()
    delinquency_cols = [
        "NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTime60-89DaysPastDueNotWorse",
        "NumberOfTimes90DaysLate",
    ]
    if all(c in out.index for c in delinquency_cols):
        out["TotalPastDueEvents"] = out[delinquency_cols].sum()
        out["HasAnyPastDue"] = int(out["TotalPastDueEvents"] > 0)
    if "NumberOfTimes90DaysLate" in out.index and "Has90DaysLate" in out.index:
        out["Has90DaysLate"] = int(out["NumberOfTimes90DaysLate"] > 0)
    if "NumberOfDependents" in out.index and "HasDependents" in out.index:
        out["HasDependents"] = int(out["NumberOfDependents"] > 0)
    if "NumberOfOpenCreditLinesAndLoans" in out.index and "NumberRealEstateLoansOrLines" in out.index and "CreditLinesPerRealEstateLoan" in out.index:
        out["CreditLinesPerRealEstateLoan"] = out["NumberOfOpenCreditLinesAndLoans"] / (1 + out["NumberRealEstateLoansOrLines"])

    log_cols = [
        "RevolvingUtilizationOfUnsecuredLines",
        "DebtRatio",
        "MonthlyIncome",
        "NumberOfOpenCreditLinesAndLoans",
        "NumberRealEstateLoansOrLines",
        "TotalPastDueEvents",
    ]
    for col in log_cols:
        log_col = f"{col}_log1p"
        if col in out.index and log_col in out.index:
            out[log_col] = np.log1p(max(float(out[col]), 0.0))
    return out


def score_single_row(row: pd.Series) -> float:
    row = recompute_engineered_features_for_row(row)
    row_df = pd.DataFrame([row[FEATURES].astype(float).values], columns=FEATURES)
    row_scaled = scaler.transform(row_df)
    return float(predict_model_score(audit_model, audit_family, row_scaled)[0])


def find_counterfactual(
    original_row: pd.Series,
    desired_class: int,
    max_steps: int = 4,
) -> Dict:
    """
    Búsqueda greedy de contrafactual.

    desired_class=0: buscamos bajar el score por debajo del threshold.
    desired_class=1: buscamos subir el score por encima del threshold.
    """
    current = original_row.copy()
    current_score = score_single_row(current)
    current_pred = int(current_score >= audit_threshold)

    if current_pred == desired_class:
        return {
            "found": True,
            "reason": "El caso ya pertenece a la clase deseada.",
            "original_score": current_score,
            "counterfactual_score": current_score,
            "changes": [],
            "counterfactual_row": current,
        }

    changes = []

    for step in range(max_steps):
        best_candidate = None
        best_score = current_score
        best_feature = None
        best_value = None

        for feature in MUTABLE_FEATURES:
            original_value = current[feature]
            for candidate_value in feature_quantiles[feature]:
                candidate = current.copy()
                candidate[feature] = candidate_value
                candidate = recompute_engineered_features_for_row(candidate)
                candidate_score = score_single_row(candidate)
                candidate_pred = int(candidate_score >= audit_threshold)

                # Si encontramos ya el cambio que cruza, devolvemos.
                if candidate_pred == desired_class:
                    new_changes = changes + [{
                        "feature": feature,
                        "old_value": float(original_value),
                        "new_value": float(candidate_value),
                        "old_score": float(current_score),
                        "new_score": float(candidate_score),
                    }]
                    return {
                        "found": True,
                        "reason": "Se encontró contrafactual.",
                        "original_score": float(score_single_row(original_row)),
                        "counterfactual_score": float(candidate_score),
                        "changes": new_changes,
                        "counterfactual_row": candidate,
                    }

                # Si no cruza, elegimos el cambio que acerca más al objetivo.
                if desired_class == 0:
                    improves = candidate_score < best_score
                else:
                    improves = candidate_score > best_score

                if improves:
                    best_candidate = candidate
                    best_score = candidate_score
                    best_feature = feature
                    best_value = candidate_value

        if best_candidate is None:
            break

        changes.append({
            "feature": best_feature,
            "old_value": float(current[best_feature]),
            "new_value": float(best_value),
            "old_score": float(current_score),
            "new_score": float(best_score),
        })
        current = best_candidate
        current_score = best_score

    return {
        "found": False,
        "reason": "No se encontró contrafactual dentro del número máximo de cambios.",
        "original_score": float(score_single_row(original_row)),
        "counterfactual_score": float(current_score),
        "changes": changes,
        "counterfactual_row": current,
    }

In [ ]:
# Generamos contrafactuales para los casos locales.
counterfactual_rows = []

for idx in local_indices:
    row = X.loc[idx]
    original_score = score_single_row(row)
    original_pred = int(original_score >= audit_threshold)
    desired = 1 - original_pred

    cf = find_counterfactual(row, desired_class=desired, max_steps=4)

    counterfactual_rows.append({
        "index": idx,
        "true_class": int(y.loc[idx]),
        "original_prediction": original_pred,
        "desired_prediction": desired,
        "found": cf["found"],
        "original_score": cf["original_score"],
        "counterfactual_score": cf["counterfactual_score"],
        "n_changes": len(cf["changes"]),
        "reason": cf["reason"],
        "changes": cf["changes"],
    })

counterfactual_summary = pd.DataFrame(counterfactual_rows)
counterfactual_summary[["index", "true_class", "original_prediction", "desired_prediction", "found", "original_score", "counterfactual_score", "n_changes", "reason"]]

In [ ]:
# Mostramos cambios concretos de cada contrafactual.
for row in counterfactual_rows:
    print("\n====================================")
    print("Cliente index:", row["index"])
    print("Clase real:", row["true_class"])
    print("Predicción original:", row["original_prediction"])
    print("Predicción deseada:", row["desired_prediction"])
    print("Encontrado:", row["found"])
    print("Score original:", round(row["original_score"], 4), "-> Score contrafactual:", round(row["counterfactual_score"], 4))

    if row["changes"]:
        display(pd.DataFrame(row["changes"]))
    else:
        print("Sin cambios propuestos.")

counterfactual_summary.to_csv(XAI_DIR / f"counterfactual_summary_{AUDIT_SCENARIO}.csv", index=False)

## 7. Cómo explicar una denegación a un cliente

La explicación debe ser útil pero cuidadosa. No conviene decir simplemente “el modelo lo decidió”. Una explicación responsable podría tener esta estructura:

1. **Factores principales**: indicar las variables que más han contribuido a la decisión.
2. **Comparación con perfiles aprobados**: explicar si el cliente está en rangos de mayor riesgo.
3. **Contrafactual accionable**: si procede, señalar qué cambios mejorarían la decisión.
4. **Evitar causalidad falsa**: SHAP y los subrogados explican el modelo, no demuestran causalidad económica.

Ejemplo de redacción:

> La solicitud ha sido clasificada como de mayor riesgo principalmente por el historial reciente de retrasos de pago, el nivel de utilización de líneas de crédito y la relación entre deuda mensual e ingresos. En perfiles similares, reducir la utilización de crédito y mejorar el historial de pagos reduce de forma significativa la probabilidad estimada de morosidad grave.

Esta explicación debe adaptarse al caso concreto usando SHAP local y el contrafactual encontrado.

## 8. Conclusiones de auditoría

Puntos que deberías llevar al notebook final o memoria:

1. El modelo de caja negra puede auditarse mediante técnicas post-hoc.
2. El árbol subrogado ofrece reglas aproximadas, pero no sustituye al modelo original.
3. SHAP permite identificar variables influyentes global y localmente.
4. Los contrafactuales ayudan a convertir una explicación técnica en una explicación accionable.
5. La política con coste `FN=10` suele ser más conservadora: acepta más falsos positivos para reducir falsos negativos.
6. La interpretabilidad debe evaluarse junto con el coste: un modelo más preciso puede ser menos transparente, por eso la auditoría es obligatoria.